# Human review coordinator: preparation, progress, and results

**For the person coordinating the independent review.** Use this notebook to check that the
frozen packet is ready, assign the two reviewer notebooks, monitor completion counts, and
summarize the submitted results after both reviewers finish.

Individual ratings are entered in [reviewer_1.ipynb](reviewer_1.ipynb) and
[reviewer_2.ipynb](reviewer_2.ipynb). This notebook does not select a reviewer, display articles
for scoring, or edit either person's answer CSV.

### Coordinator workflow

1. Complete the preparation checklist and locate the frozen packet folder.
2. Give each person their assigned notebook and files; preserve independent judgments.
3. Check completion counts without displaying individual ratings.
4. After both reviewers submit, explicitly enable the combined summary.
5. Write the team's interpretation and hand the outputs back to notebook 03 for PR3.

Default execution only checks available inputs. Combined summaries and interpretation saves
remain off until enabled. Missing prerequisites are reported as pending; invalid files raise
an error for investigation. Existing packets and original answers are never regenerated here.

## Preparation checklist

Before distributing the review:

- Notebook 01 has reserved the same 100 evaluation article IDs for both pipelines, separately
  from development/naming evidence. These articles are not held out from unsupervised fitting.
- The team has selected the modern configuration and confirmed its article-supported topic labels.
- Laura has supplied matching LDA exports, confirmed labels, and the shared reservation attestation.
- Notebook 03 has generated the frozen packet, its manifest/status, and two blank reviewer CSVs.
- Two people have been assigned to reviewer 1 and reviewer 2. Each person reviews both models'
  assignments for the same articles: 200 article–model items per person.

If the packet does not exist, complete these steps in analysis notebooks 01–03 first. Do not
invent missing labels, ratings, or a model choice. The coordinator does not need to refit models
from this notebook.

Use `uv run jupyter lab` from the repository environment. Models, article IDs, and labels must
remain fixed during review; label or sample changes require an explicit new evaluation rather
than overwriting existing independent answers.

In [1]:
from pathlib import Path
import json
import sys
import pandas as pd
from IPython.display import display

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
ROOT = REPO / "final-project/code/modern-methods"
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT.parent))
from src.reviewer import load_session
from shared.human_review import summarize_reviews

## Step 1 — locate the frozen packet

Normally leave `PACKET_FOLDER = None` to follow the selected modern run. If the team uses a
separate local folder, enter its absolute path. This folder must contain the common packet,
its freeze/status files, and the two independent reviewer CSVs.

The default path remains
`final-project/code/modern-methods/Results/comparison/<selected-run>/human_review/`.
Renaming this notebook does not move data or change where reviewers save their answers.

In [2]:
# OPTIONAL: use an absolute path only if the team has a separate packet folder.
PACKET_FOLDER = None       # Example: Path("/absolute/path/to/human_review")
REVIEW_FOLDER = None
COMMON_FILES = ["review_packet.csv", "review_freeze.json", "review_status.json"]
REVIEWER_FILES = ["reviewer_1.csv", "reviewer_2.csv"]

if PACKET_FOLDER is not None:
    REVIEW_FOLDER = Path(PACKET_FOLDER).expanduser().resolve()
else:
    selection_file = ROOT / "Results/modern/selection.json"
    selection = json.loads(selection_file.read_text()) if selection_file.exists() else {}
    if selection.get("run_id"):
        REVIEW_FOLDER = ROOT / "Results/comparison" / selection["run_id"] / "human_review"

if REVIEW_FOLDER is None:
    print("Pending: finish model selection and labels, then generate the joint review packet in notebook 03.")
else:
    missing = [name for name in COMMON_FILES + REVIEWER_FILES if not (REVIEW_FOLDER / name).exists()]
    print("Packet folder:", REVIEW_FOLDER)
    if missing:
        print("Pending files:", ", ".join(missing))
        print("Use notebook 03 to prepare the packet, or collect the missing original reviewer files.")
    else:
        print("All required files are present. Run Step 3 to validate them and check completion.")

Pending: finish model selection and labels, then generate the joint review packet in notebook 03.


## Step 2 — distribute the individual notebooks

| Person | Notebook | Editable answer file |
| --- | --- | --- |
| Reviewer 1 | [reviewer_1.ipynb](reviewer_1.ipynb) | `reviewer_1.csv` |
| Reviewer 2 | [reviewer_2.ipynb](reviewer_2.ipynb) | `reviewer_2.csv` |

Each person's ID is already configured. Give them the common packet, `review_freeze.json`,
`review_status.json`, and **their own** answer CSV; they do not need the other person's scores.
Ask them to use separate kernels/personal working copies and to work independently.

Both notebooks contain the same rubric: fit 1–5; a separate article-irrelevance flag; a
label-ambiguity flag; and optional explanatory notes. For topic −1, the numeric rating stays
blank and the qualitative fields are completed. The model identities are visible, so the
review is not blinded. Ratings should reflect article–label fit, not sentiment or agreement
with the article's content.

Personal working copies named `reviewer_1_workbook.ipynb` or `reviewer_2_workbook.ipynb` inside
the packet folder are ignored by Git. Collect only the original answer CSVs for analysis.
Do not discuss differences until both reviewers finish, and do not replace independent
ratings with consensus scores.

## Step 3 — monitor completion without showing scores

The cell below validates each available reviewer file against the frozen packet and displays
only counts. It does not expose individual scores, article text, or reviewer notes.
Missing files remain pending. An item is complete when its required numeric rating (if
assigned) and both qualitative flags are present; explanatory notes are optional.

Progress checks do not update either answer file. If a reviewer reports a problem with an
article or label, document it and investigate rather than editing the frozen evaluation.

In [3]:
PROGRESS = None
if REVIEW_FOLDER is None or any(not (REVIEW_FOLDER / name).exists() for name in COMMON_FILES):
    print("Pending: locate a complete frozen packet in Step 1.")
else:
    progress_rows = []
    for reviewer in ["reviewer_1", "reviewer_2"]:
        if not (REVIEW_FOLDER / f"{reviewer}.csv").exists():
            progress_rows.append(dict(reviewer=reviewer, completed=None, total=None, remaining=None, status="file pending"))
            continue
        session = load_session(REVIEW_FOLDER, reviewer)
        total = len(session["packet"])
        completed = int(session["complete"].sum())
        progress_rows.append(dict(reviewer=reviewer, completed=completed, total=total,
                                  remaining=total-completed, status="complete" if completed == total else "in progress"))
    PROGRESS = pd.DataFrame(progress_rows)
    display(PROGRESS)
    print("No individual ratings are displayed or modified. Confirm both submissions before enabling Step 4.")

Pending: locate a complete frozen packet in Step 1.


## Step 4 — summarize both completed submissions

Set both switches to `True` only after both reviewers submit their independent files.
The summary rereads and validates the files; a progress count from an earlier session is
not treated as proof that current files are complete. Incomplete ratings return a pending
message. Invalid identities, outlier ratings, or values raise an error.

| Output | Meaning |
| --- | --- |
| `sample_articles`, `rated_articles` | Sample size and numeric-rating denominator for each model. |
| `unassigned_articles` | Items excluded from numeric ratings because topic = −1. |
| `mean_1`, `mean_2` | Each reviewer's mean assigned-article fit score. |
| `exact_agreement`, `within_one_agreement` | Fractions of paired ratings that match or differ by at most one point. |
| `weighted_kappa` | Quadratic-weighted agreement beyond chance; larger disagreements count more. |
| `kappa_ci_low`, `kappa_ci_high` | 95% interval from 1,000 paired-article bootstrap resamples. |
| `valid_bootstrap_samples`, `kappa_status` | Degenerate resample count and explicit explanation of undefined agreement. |

Agreement measures consistency between reviewers, not model accuracy. Report numeric-rating
coverage alongside quality scores, and do not interpret undefined kappa as zero agreement.
The separate supplemental `outlier_review.csv` is qualitative and is not added to the
100-article agreement sample.

This step writes aggregate results but preserves the two original answer files. The merged
`review_examples.csv` contains individual judgments and stays ignored by Git; inspect it
only after independent submissions are complete.

In [4]:
# COORDINATOR ONLY: leave both False until both reviewers have submitted independently.
BOTH_REVIEWERS_FINISHED = False
RUN_TEAM_SUMMARY = False
TEAM_SUMMARY = None

if BOTH_REVIEWERS_FINISHED and RUN_TEAM_SUMMARY:
    if REVIEW_FOLDER is None:
        print("Pending: choose the frozen packet folder in Step 1.")
    else:
        missing = [name for name in COMMON_FILES + REVIEWER_FILES if not (REVIEW_FOLDER / name).exists()]
        if missing:
            print("Pending files:", ", ".join(missing))
        else:
            TEAM_SUMMARY = summarize_reviews(REVIEW_FOLDER, bootstrap_samples=1000, seed=42)
            display(TEAM_SUMMARY)
            if isinstance(TEAM_SUMMARY, pd.DataFrame):
                print("Saved human_evaluation.csv and rating_distributions.csv in", REVIEW_FOLDER)
                print("Original reviewer files are unchanged. Continue to the interpretation and handoff steps.")
else:
    print("Team summary not run. Independent reviewer scores have not been compared.")

Team summary not run. Independent reviewer scores have not been compared.


## Step 5 — write the team's interpretation

Fill in the notes below after the completed numerical summary is available:

- How well do the labels describe assigned articles? Consider both reviewers and score distributions.
- Where did reviewers disagree? Were labels unclear/broad, or were articles multi-topic?
- What do unassigned and irrelevant cases show? Use IDs and brief paraphrases, not copied articles.
- How do conditional coverage, two team reviewers, a small sample, and visible model identities
  limit the findings? The sample is separate from development review, not from model fitting.

Leave unsupported conclusions explicit rather than inventing findings. The optional save
writes human-authored `team_interpretation.csv`; it does not edit the PR3 manuscript or change
original independent ratings.

In [5]:
# Replace the empty strings with your team's interpretation of the completed results.
TEAM_NOTES = {
    "article_topic_fit": "",
    "reviewer_disagreements": "",
    "unassigned_or_irrelevant_examples": "",
    "limitations_and_report_wording": "",
}
SAVE_TEAM_NOTES = False

if SAVE_TEAM_NOTES:
    if not isinstance(TEAM_SUMMARY, pd.DataFrame):
        raise ValueError("Run a complete team summary before saving interpretation notes.")
    if any(not text.strip() for text in TEAM_NOTES.values()):
        raise ValueError("Fill in each interpretation field, or state that no conclusion is supported.")
    notes_table = pd.DataFrame(TEAM_NOTES.items(), columns=["section", "human_interpretation"])
    notes_table.to_csv(REVIEW_FOLDER / "team_interpretation.csv", index=False)
    display(notes_table)
else:
    print("No interpretation notes saved. Fill these in after both reviewers finish.")

No interpretation notes saved. Fill these in after both reviewers finish.


## Step 6 — hand off the results for PR3

1. Preserve the frozen packet and both original reviewer CSVs in their existing folder.
2. Use `human_evaluation.csv`, `rating_distributions.csv`, and the team's interpretation notes
   for the human-evaluation results. Include sample sizes, outlier coverage, uncertainty, and limitations.
3. Rerun [analysis notebook 03](../modern-methods/notebooks/03_evaluation_and_comparison.ipynb)
   to refresh the comparison outputs and PR3 evidence checklist.
4. Apply manuscript updates manually; this notebook never writes `pr3-submission.tex`.

Keep full-text packets, original ratings, and merged review examples local. If you share this
notebook as a blank template, reset editable notes and save switches and clear executed outputs.